# Fine-tuning Audio8-TTS-Preview-0.1b sur le malgache

**Avant de commencer**
1. Exécution > Modifier le type d'exécution > **GPU** (L4 ou A100 conseillé ; le T4 fonctionne en fp32, plus lent).
2. Exécute la cellule **[1]**, puis **Exécution > Redémarrer la session** (obligatoire : `transformers` doit être < 5).
3. Après le redémarrage, **saute la cellule [1]** et lance dans l'ordre : 2, 2b, 3, **3b**, 4a, 4b, 4c, 5, 6, 6b, 7, 8, 9.
4. Remplace `HF_USER` dans la cellule 2 par ton nom d'utilisateur Hugging Face.

**Encodage fait une seule fois** : la cellule 3b restaure depuis ton dépôt Hub privé (`CODES_REPO`) les codes déjà encodés.
- Si elle indique que `codes.pt` est présent et que le nombre d'heures te suffit : **saute 4a et 4b**, lance seulement **4c**, puis 5 et la suite.
- Sinon, 4a et 4b encodent la suite (reprise automatique, chaque part est envoyée sur le Hub au fur et à mesure), puis 4c fusionne et envoie `codes.pt`.
- Pour encoder plus d'heures plus tard, augmente `MAX_HOURS` : seules les heures manquantes sont encodées.

Si un ancien dossier `/content/work/parts` existe d'un run avec erreurs mémoire, supprime-le (`!rm -rf /content/work/parts`).
**Ne lance pas la cellule 7 avant d'avoir vérifié que la cellule 6b affiche `CPU OK`.**

# **[1] Installation des dépendances**

In [8]:
!pip install -q "torch>=2.5.0" "torchaudio>=2.5.0" "transformers>=4.57.0,<5" "soundfile>=0.12" "safetensors>=0.4" accelerate pandas pyarrow

# >>> Puis : Exécution > Redémarrer la session, et saute cette cellule ensuite.

# **[2] Imports et configuration**

In [9]:
import io, json, os, random, re, shutil, tarfile, unicodedata
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

# Erreurs CUDA synchrones : la ligne indiquée dans la trace est alors la vraie coupable.
# (plus lent : mettre False une fois l'entraînement validé). Doit être défini AVANT "import torch".
DEBUG_CUDA = False
if DEBUG_CUDA:
    os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")   # moins de fragmentation

import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torch.nn.functional as F
from huggingface_hub import hf_hub_download, snapshot_download
from torch.utils.checkpoint import checkpoint
import torchaudio
from tqdm.auto import tqdm
from transformers import AutoModel, AutoProcessor, Trainer, TrainingArguments

MODEL_ID = "Audio8/Audio8-TTS-Preview-0.1b"
DATA_REPO = "Professor/malagasy-speech-data"

WORK = "/content/work"                       # cache des codes codec
OUT = "/content/audio8_malagasy_sft"         # checkpoints + modèle final
MAX_SAMPLES = None                           # ex: 300 pour un test rapide
MIN_SEC, MAX_SEC = 1.0, 20.0                 # filtre de durée
MAX_LENGTH = 2048                            # contexte max du modèle

# Vitesse de l'encodage codec
MAX_HOURS = 20      # ex: 20 -> s'arrête après ~20 h d'audio encodé (recommandé pour un premier run)
ENC_BATCH = 16          # baisser à 8 en cas de saturation mémoire GPU
CHUNK = 128             # nb d'échantillons décodés à la fois (limite la RAM)
NUM_WORKERS = 4         # threads de décodage FLAC

EPOCHS = 5
LEARNING_RATE = 2e-5
BATCH_SIZE = 4
GRAD_ACCUM = 4

# Peu de données (< 5 h) : mettre True pour n'entraîner que la branche lente
FREEZE_FAST_AR = False

# Hugging Face Hub (le disque Colab est éphémère)
HF_USER = "Starfish1337"                          # <- ton nom d'utilisateur HF
HF_REPO = f"{HF_USER}/audio8-tts-malagasy"    # modèle final
PUSH_CHECKPOINTS = False                      # True : pousse aussi les checkpoints (repo "-ckpt")
UPLOAD_FINAL = True                           # envoie le modèle final sur le Hub
PERSIST_CODES = True                          # sauvegarde persistante des codes codec sur le Hub (encodage fait UNE fois)
CODES_REPO = f"{HF_USER}/audio8-malagasy-codes"   # dépôt Hub PRIVÉ de type "dataset"
RESUME_FROM = None                            # chemin d'un checkpoint pour reprendre, sinon None

os.makedirs(WORK, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
use_bf16 = device == "cuda" and torch.cuda.get_device_capability()[0] >= 8   # T4 (7.5) : pas de vrai bf16 -> fp32
enc_dtype = torch.bfloat16 if use_bf16 else torch.float32
random.seed(0)

# **[2b] Connexion Hugging Face (nécessaire pour l'envoi sur le Hub)**

In [10]:
if UPLOAD_FINAL or PUSH_CHECKPOINTS or PERSIST_CODES:
    from huggingface_hub import notebook_login
    notebook_login()

# **[3] Lecture du manifest**

In [11]:
df = pd.read_parquet(hf_hub_download(DATA_REPO, "manifest.parquet", repo_type="dataset"))
print(df.columns.tolist(), len(df))

TEXT_COL = next(
    (c for c in ["text", "transcription", "transcript", "sentence", "normalized_text"] if c in df.columns),
    None,
)
if TEXT_COL is None:
    raise ValueError(f"Colonne de texte introuvable, colonnes = {df.columns.tolist()} -> renseigne TEXT_COL")
print("Colonne de texte :", TEXT_COL)
if MAX_SAMPLES:
    df = df.head(MAX_SAMPLES)


def clean_text(value) -> str:
    text = "".join(
        ch if ch.isspace() else "" if unicodedata.category(ch).startswith("C") else ch
        for ch in str(value)
    )
    return re.sub(r"\s+", " ", text).strip()

['key', 'shard', 'text', 'duration', 'dataset_id', 'source', 'split', 'speaker_id', 'gender', 'domain', 'snr', 'dbfs', 'clip_ratio', 'sil_ratio', 'has_disfluency'] 34293
Colonne de texte : text


# **[3b] Sauvegarde persistante : restauration depuis le Hub (à lancer avant 4a)**

In [12]:
from huggingface_hub import HfApi

api = HfApi()
upload_pool = ThreadPoolExecutor(max_workers=1)        # envois en arrière-plan, sans ralentir l'encodage
pending = []


def persist(path_in_repo, local_path, background=True):
    """Envoie un fichier dans le dépôt privé CODES_REPO (sauvegarde persistante)."""
    if not PERSIST_CODES:
        return

    def _up():
        try:
            api.upload_file(path_or_fileobj=local_path, path_in_repo=path_in_repo,
                            repo_id=CODES_REPO, repo_type="dataset")
        except Exception as exc:
            print("envoi Hub échoué :", path_in_repo, type(exc).__name__, str(exc)[:100])

    if background:
        pending.append(upload_pool.submit(_up))
    else:
        _up()


def wait_uploads():
    for fut in pending:
        fut.result()
    pending.clear()


def restore_codes():
    """Récupère codes.pt, la référence vocale et les parts depuis le Hub, si une sauvegarde existe."""
    if not PERSIST_CODES:
        return
    api.create_repo(CODES_REPO, repo_type="dataset", private=True, exist_ok=True)
    files = [f for f in api.list_repo_files(CODES_REPO, repo_type="dataset")
             if f in ("codes.pt", "ref.wav", "ref.txt") or f.startswith("parts/")]
    todo = [f for f in files if not os.path.exists(f"{WORK}/{f}")]
    for f in tqdm(todo, desc="restauration depuis le Hub"):
        hf_hub_download(CODES_REPO, f, repo_type="dataset", local_dir=WORK)
    print(f"Hub ({CODES_REPO}) : {len(files)} fichiers, {len(todo)} restaurés")
    if os.path.exists(f"{WORK}/codes.pt"):
        print("codes.pt présent -> tu peux sauter 4a et 4b et lancer directement 4c "
              "(sauf si tu veux encoder plus d'heures : augmente alors MAX_HOURS).")


restore_codes()

restauration depuis le Hub: 0it [00:00, ?it/s]

Hub (Starfish1337/audio8-malagasy-codes) : 17 fichiers, 0 restaurés


# **[4a] Préparation de l'encodage : codec + fonctions (à lancer une fois)**

In [13]:
import gc, glob, math, queue, threading, time
from pathlib import Path

PARTS = f"{WORK}/parts"                        # résultats intermédiaires (1 fichier par "part")
os.makedirs(PARTS, exist_ok=True)
CODES_PATH = f"{WORK}/codes.pt"
SAVE_EVERY = 2      # une part est sauvegardée tous les SAVE_EVERY paquets de CHUNK échantillons
QUEUE_SIZE = 3      # paquets décodés préparés à l'avance par le thread CPU
ENC_BATCH_SEC = 160  # budget par batch, en secondes d'audio (réduit automatiquement si le GPU sature)
FRAME = 2048        # échantillons audio par frame codec

if os.environ.get("CUDA_LAUNCH_BLOCKING") == "1":
    print("ATTENTION : CUDA_LAUNCH_BLOCKING=1 (DEBUG_CUDA=True) ralentit fortement l'encodage. "
          "Mets DEBUG_CUDA = False (cellule 2) et redémarre la session pour cette étape.")
print("Device :", device, "| dtype codec :", enc_dtype)
if device != "cuda":
    print("ATTENTION : pas de GPU -> encodage extrêmement lent.")

# Même procédure que audio8_tts_prepare.py : le modèle reste sur CPU, seul le codec va sur GPU.
# Cette instance est supprimée en 4c pour que le codec ne se retrouve pas dans le modèle entraîné.
enc = AutoModel.from_pretrained(MODEL_ID, trust_remote_code=True, dtype=enc_dtype).eval()
enc.load_codec(device=device, dtype=enc_dtype)
codec_sr = int(enc.config.codec_sample_rate)
n_cb = int(enc.config.num_codebooks)

# Codes déjà en cache (ex. un précédent run de 10 h) : on ne les ré-encode pas
existing = torch.load(CODES_PATH) if os.path.exists(CODES_PATH) else []
known_ids = {r["id"] for r in existing}
existing_sec = sum(r["codes"].shape[1] for r in existing) * FRAME / codec_sr
print(f"Déjà en cache : {len(existing)} échantillons ({existing_sec / 3600:.2f} h)")

# Si le manifest contient la durée, on écarte les clips hors limites SANS les lire ni les décoder
DUR_COL = next((c for c in ["duration", "duration_s", "audio_duration", "dur", "length_s"]
                if c in df.columns and pd.api.types.is_numeric_dtype(df[c])), None)
if DUR_COL and float(df[DUR_COL].median()) > 200:      # probablement des millisecondes -> on n'y touche pas
    DUR_COL = None
print("Colonne de durée du manifest :", DUR_COL)

resamplers = {}                                        # noyaux de rééchantillonnage mis en cache (GPU)


def to_codec_rate(wave, sr):
    wave = wave.to(device)
    if sr == codec_sr:
        return wave
    if sr not in resamplers:
        resamplers[sr] = torchaudio.transforms.Resample(sr, codec_sr).to(device)
    return resamplers[sr](wave)


@torch.inference_mode()
def encode(waves):                                     # waves : tenseurs 1-D déjà sur le device
    lengths = torch.tensor([w.numel() for w in waves], dtype=torch.long, device=device)
    values = torch.zeros((len(waves), 1, int(lengths.max())), dtype=torch.float32, device=device)
    for i, w in enumerate(waves):
        values[i, 0, : w.numel()] = w
    codes, code_lengths = enc.encode_audio(values, lengths)
    codes = codes.to(torch.int16).cpu().numpy()        # un seul transfert GPU->CPU par batch
    out = []
    for i, n in enumerate(code_lengths.cpu().tolist()):
        c = codes[i, :, : int(n)]
        assert c.shape[0] == n_cb and c.shape[1] > 0, c.shape
        out.append(torch.from_numpy(np.ascontiguousarray(c)))
    return out


cur = {"sec": ENC_BATCH_SEC}


def make_batches(items):
    """Regroupe des clips triés par longueur croissante, avec un budget de secondes d'audio par batch
    (padding compris) : peu de gros clips ou beaucoup de petits, pour une mémoire GPU stable."""
    budget = cur["sec"] * codec_sr
    batches, batch = [], []
    for it in items:
        if batch and (len(batch) + 1) * it[2].numel() > budget:     # `it` est le plus long du batch
            batches.append(batch)
            batch = []
        batch.append(it)
    if batch:
        batches.append(batch)
    return batches


def encode_safe(batch, retry=True):
    """Encode un batch ; en cas d'erreur (mémoire GPU...), libère la mémoire puis coupe en deux."""
    err = None
    try:
        return encode([b[2] for b in batch])
    except Exception as exc:
        msg = str(exc)
        if "device-side assert" in msg:
            raise                                      # contexte CUDA corrompu : inutile d'insister
        is_oom = isinstance(exc, torch.cuda.OutOfMemoryError) or "out of memory" in msg.lower()
        err = (type(exc).__name__, msg[:100], is_oom)
    # IMPORTANT : on est SORTI du bloc except. Tant qu'on restait dedans, la trace de l'exception
    # gardait en mémoire les tenseurs du batch échoué, ce qui faisait échouer tous les essais suivants.
    gc.collect()
    torch.cuda.empty_cache()
    if len(batch) == 1:
        if retry:                                      # un clip seul doit tenir une fois la mémoire libérée
            return encode_safe(batch, retry=False)
        print("skip", batch[0][0], *err[:2])
        return [None]
    if err[2]:
        cur["sec"] = max(20, cur["sec"] * 0.6)
        print(f"mémoire GPU saturée -> budget réduit à {cur['sec']:.0f} s d'audio par batch")
    mid = len(batch) // 2
    return encode_safe(batch[:mid]) + encode_safe(batch[mid:])


def decode(job):                                       # exécuté dans un thread
    key, raw, text = job
    try:
        arr, sr = sf.read(io.BytesIO(raw), dtype="float32", always_2d=True)
    except Exception as exc:
        print("skip", key, type(exc).__name__, exc)
        return None
    wave = arr.mean(axis=1)
    dur = len(wave) / sr
    if not (MIN_SEC <= dur <= MAX_SEC) or not np.isfinite(wave).all():
        return None
    return key, text, torch.from_numpy(wave), sr


pool = ThreadPoolExecutor(max_workers=NUM_WORKERS)


def producer(q, stop):
    """Thread CPU : télécharge, lit et décode les shards PENDANT que le GPU encode."""
    def put(item):
        while not stop.is_set():
            try:
                q.put(item, timeout=1)
                return True
            except queue.Full:
                pass
        return False

    try:
        size = SAVE_EVERY * CHUNK
        for shard, group in df.groupby("shard"):
            rows = list(group.itertuples())
            if DUR_COL:
                rows = [r for r in rows if MIN_SEC <= float(getattr(r, DUR_COL)) <= MAX_SEC]
            todo = []
            for g, start in enumerate(range(0, len(rows), size)):
                pid = f"{Path(str(shard)).stem}-{g:04d}"
                if not os.path.exists(f"{PARTS}/{pid}.pt"):
                    todo.append((pid, rows[start : start + size]))
            if not todo:
                continue
            shard_path = hf_hub_download(DATA_REPO, f"shards/{shard}", repo_type="dataset")
            with tarfile.open(shard_path) as tar:
                members = {m.name: m for m in tar.getmembers() if m.isfile()}
                for pid, grp in todo:
                    for ci in range(0, len(grp), CHUNK):
                        if stop.is_set():
                            return
                        jobs = []
                        for row in grp[ci : ci + CHUNK]:
                            if str(row.key) in known_ids:
                                continue
                            m = members.get(f"{row.key}.flac")
                            if m is None:
                                continue
                            try:
                                text = clean_text(getattr(row, TEXT_COL))
                                if text:
                                    jobs.append((str(row.key), tar.extractfile(m).read(), text))
                            except Exception as exc:
                                print("skip", row.key, type(exc).__name__, exc)
                        decoded = [d for d in pool.map(decode, jobs) if d is not None]
                        last = ci + CHUNK >= len(grp)
                        if not put((pid, last, decoded)):
                            return
    except BaseException as exc:                       # on transmet l'erreur au thread principal
        put(("__error__", exc))
    finally:
        put(None)


def done_sec():
    """Échantillons uniques et heures déjà encodés (codes.pt + parts), sans double comptage."""
    frames = {r["id"]: r["codes"].shape[1] for r in existing}
    for f in glob.glob(f"{PARTS}/*.pt"):
        for r in torch.load(f)["records"]:
            frames[r["id"]] = r["codes"].shape[1]
    return len(frames), sum(frames.values()) * FRAME / codec_sr


def status():
    n, sec = done_sec()
    print(f"{len(glob.glob(f'{PARTS}/*.pt'))} parts | {n} échantillons uniques | {sec / 3600:.2f} h encodées")


status()

Device : cuda | dtype codec : torch.float32


/usr/local/lib/python3.13/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Déjà en cache : 0 échantillons (0.00 h)
Colonne de durée du manifest : duration
17 parts | 4210 échantillons uniques | 20.34 h encodées


# **[4b] Encodage avec suivi en direct (reprenable, relançable)**

In [14]:
# Reprenable : on peut interrompre (bouton stop) et relancer, les parts déjà faites sont ignorées.
# Ne change pas MIN_SEC / MAX_SEC / MAX_SAMPLES entre deux lancements (les parts seraient décalées) :
# si tu les changes, supprime d'abord le dossier PARTS.
def parts_sec():
    return sum(torch.load(f)["sec"] for f in glob.glob(f"{PARTS}/*.pt"))


def save_part(pid, records_, sec_):
    tmp = f"{PARTS}/{pid}.tmp"
    torch.save({"records": records_, "sec": sec_}, tmp)
    os.replace(tmp, f"{PARTS}/{pid}.pt")                # écriture atomique


base_sec = done_sec()[1]
limit_sec = MAX_HOURS * 3600 if MAX_HOURS else None
print(f"Départ : {base_sec / 3600:.2f} h déjà encodées | objectif : {MAX_HOURS or 'tout le dataset'} h")

if limit_sec and base_sec >= limit_sec:
    print("Objectif déjà atteint -> passe à la cellule 4c.")
else:
    stop = threading.Event()
    q = queue.Queue(maxsize=QUEUE_SIZE)
    threading.Thread(target=producer, args=(q, stop), daemon=True).start()

    new_sec, buf, buf_sec, n_ok = 0.0, [], 0.0, 0
    t0, t_wait = time.time(), 0.0
    if device == "cuda":
        torch.cuda.reset_peak_memory_stats()
    if limit_sec:
        pbar = tqdm(total=MAX_HOURS, initial=min(base_sec / 3600, MAX_HOURS), unit="h", desc="encodage",
                    bar_format="{l_bar}{bar}| {n:.2f}/{total} h [{elapsed}<{remaining}] {postfix}")
    else:
        pbar = tqdm(unit="h", desc="encodage")

    try:
        while True:
            tw = time.time()
            item = q.get()
            t_wait += time.time() - tw
            if item is None:
                break
            if len(item) == 2:
                raise item[1]
            pid, last, decoded = item

            with torch.inference_mode():
                items = [(k, t, to_codec_rate(torch.as_tensor(w), sr)) for k, t, w, sr in decoded]

            if items and not os.path.exists(f"{WORK}/ref.wav"):      # clip de référence (~6 s)
                ref = min(items, key=lambda it: abs(it[2].numel() / codec_sr - 6.0))
                sf.write(f"{WORK}/ref.wav", ref[2].float().cpu().numpy(), codec_sr)
                with open(f"{WORK}/ref.txt", "w", encoding="utf-8") as f:
                    f.write(ref[1])

            items.sort(key=lambda it: it[2].numel())                 # longueurs voisines -> moins de padding
            chunk_sec = 0.0
            for batch in make_batches(items):
                for b, c in zip(batch, encode_safe(batch)):
                    if c is not None:
                        buf.append({"id": b[0], "text": b[1], "codes": c})
                        s = b[2].numel() / codec_sr
                        buf_sec += s
                        chunk_sec += s
                        n_ok += 1
            new_sec += chunk_sec
            del items, decoded
            torch.cuda.empty_cache()                                  # limite la fragmentation

            reached = bool(limit_sec) and base_sec + new_sec >= limit_sec
            if (last or reached) and buf:
                save_part(pid, buf, buf_sec)
                persist(f"parts/{pid}.pt", f"{PARTS}/{pid}.pt")           # copie sur le Hub (arrière-plan)
                tqdm.write(f"[part {pid}] {len(buf)} échantillons, {buf_sec / 3600:.2f} h sauvegardés")
                buf, buf_sec = [], 0.0

            elapsed = time.time() - t0
            # la barre ne doit jamais dépasser son total : tqdm supprimerait alors `total` (None)
            pbar.update(min(chunk_sec / 3600, max(0.0, MAX_HOURS - pbar.n)) if limit_sec else chunk_sec / 3600)
            pbar.set_postfix(
                ok=n_ok,
                x_temps_reel=round(new_sec / elapsed),            # secondes d'audio encodées par seconde
                attente_cpu=f"{100 * t_wait / elapsed:.0f}%",     # GPU qui attend le CPU
                file=f"{q.qsize()}/{QUEUE_SIZE}",
                budget_s=round(cur["sec"]),
                gpu_Go=round(torch.cuda.max_memory_allocated() / 1e9, 1) if device == "cuda" else 0,
            )
            if reached:
                break
    except KeyboardInterrupt:
        print("Interrompu : les parts déjà sauvegardées sont conservées. Relance cette cellule pour reprendre.")
    finally:
        stop.set()
        pbar.close()
    wait_uploads() if "wait_uploads" in globals() else None          # attend la fin des envois Hub

    elapsed = time.time() - t0
    print(f"Terminé : +{new_sec / 3600:.2f} h en {elapsed / 60:.1f} min "
          f"(x{new_sec / max(elapsed, 1e-9):.0f} temps réel) | attente CPU {100 * t_wait / max(elapsed, 1e-9):.0f}%")
    print("Lecture : attente_cpu élevée (> 30 %) = le CPU limite -> augmente NUM_WORKERS ; "
          "attente_cpu proche de 0 = le GPU limite (déjà au maximum).")
    status()

Départ : 20.34 h déjà encodées | objectif : 20 h
Objectif déjà atteint -> passe à la cellule 4c.


# **[4c] Fusion des parts -> codes.pt et libération du GPU**

In [15]:
# Fusionne le cache existant + toutes les parts en codes.pt, puis libère le GPU et le disque.
# Peut être lancée seule pour simplement recharger un codes.pt existant (sans 4a ni 4b).
records, seen = [], set()
sources = [torch.load(CODES_PATH)] if os.path.exists(CODES_PATH) else []
sources += [torch.load(f)["records"] for f in sorted(glob.glob(f"{WORK}/parts/*.pt"))]
for recs in sources:
    for r in recs:
        if r["id"] not in seen:
            seen.add(r["id"])
            records.append(r)
torch.save(records, CODES_PATH)
print(f"codes.pt : {len(records)} échantillons")

if PERSIST_CODES and "persist" in globals():                 # sauvegarde persistante
    wait_uploads()
    for name in ("codes.pt", "ref.wav", "ref.txt"):
        if os.path.exists(f"{WORK}/{name}"):
            persist(name, f"{WORK}/{name}", background=False)
    print("Sauvegarde persistante : https://huggingface.co/datasets/" + CODES_REPO)

if "enc" in globals():
    del enc
if "pool" in globals():
    pool.shutdown(wait=False)
torch.cuda.empty_cache()
shutil.rmtree(                                         # les shards ne sont plus nécessaires
    os.path.expanduser("~/.cache/huggingface/hub/datasets--Professor--malagasy-speech-data"),
    ignore_errors=True,
)

codes.pt : 4210 échantillons


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  /content/work/codes.pt      :   2%|1         |  653kB / 33.6MB            

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  /content/work/ref.wav       : 100%|#########9| 1.33MB / 1.33MB            

Sauvegarde persistante : https://huggingface.co/datasets/Starfish1337/audio8-malagasy-codes


# **[5] Construction des exemples (format DualAR du pipeline officiel)**

In [16]:
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer = processor.tokenizer


def tok(*parts):
    ids = [t for p in parts for t in tokenizer.encode(p, add_special_tokens=False)]
    return torch.tensor(ids, dtype=torch.long)


def build_example(config, text, codes):
    """Entrées décalées d'un cran : la sortie à la position t prédit labels[t]."""
    codes = codes.long()
    n_cb = int(config.num_codebooks)
    prefix = tok(
        "<|im_start|>system\n", "convert the provided text to speech", "<|im_end|>\n",
        "<|im_start|>user\n", text, "<|im_end|>\n",
        "<|im_start|>assistant\n<|voice|>",
    )
    audio_tokens = codes[0] + int(config.semantic_begin_id)
    end = tok("<|im_end|>\n")
    a0, a1 = prefix.numel(), prefix.numel() + codes.shape[1]
    total = a1 + end.numel()
    if total - 1 > MAX_LENGTH:
        raise ValueError(f"exemple trop long : {total - 1} > {MAX_LENGTH}")

    full = torch.zeros((n_cb + 1, total), dtype=torch.long)
    full[0, :a0] = prefix
    full[0, a0:a1] = audio_tokens
    full[1:, a0:a1] = codes
    full[0, a1:] = end
    supervised = torch.zeros(total, dtype=torch.bool)
    supervised[a0:] = True                       # audio + <|im_end|>
    cb_supervised = torch.zeros(total, dtype=torch.bool)
    cb_supervised[a0:a1] = True                  # codebooks : uniquement sur l'audio

    input_ids = full[:, :-1].contiguous()
    labels = torch.full_like(input_ids, -100)
    ns = supervised[1:]
    labels[0, ns] = full[0, 1:][ns]
    nc = cb_supervised[1:]
    labels[1:, nc] = full[1:, 1:][:, nc]
    return {"input_ids": input_ids, "labels": labels}


def collate(examples, pad_token_id):
    width = max(e["input_ids"].shape[1] for e in examples)
    ids, labs, masks = [], [], []
    for e in examples:
        x, y = e["input_ids"], e["labels"]
        n = x.shape[1]
        if width > n:
            x = F.pad(x, (0, width - n), value=0)
            x[0, n:] = pad_token_id
            y = F.pad(y, (0, width - n), value=-100)
        m = torch.zeros(width, dtype=torch.long)
        m[:n] = 1
        ids.append(x); labs.append(y); masks.append(m)
    return {"input_ids": torch.stack(ids), "attention_mask": torch.stack(masks), "labels": torch.stack(labs)}


class TTSDataset(torch.utils.data.Dataset):
    def __init__(self, rows, config):
        self.rows, self.config = rows, config

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        r = self.rows[i]
        return build_example(self.config, r["text"], r["codes"])

processor_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

processing_arktts.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/Audio8/Audio8-TTS-Preview-0.1b:
- processing_arktts.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

# **[6] Modèle d'entraînement (poids fp32, autocast bf16 par le Trainer)**

In [17]:
model = AutoModel.from_pretrained(MODEL_ID, trust_remote_code=True, dtype=torch.float32)
model.config.use_gradient_checkpointing = True

hours = sum(r["codes"].shape[1] for r in records) * 2048 / model.config.codec_sample_rate / 3600
print(f"Durée totale encodée : {hours:.2f} h")

if FREEZE_FAST_AR:
    for name, p in model.named_parameters():
        if name.startswith(("fast_project_in.", "fast_embeddings.", "fast_layers.", "fast_norm.", "fast_output.")):
            p.requires_grad = False
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Paramètres entraînables : {trainable / 1e6:.1f}M / {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

random.shuffle(records)
n_eval = max(20, int(0.02 * len(records)))
train_ds = TTSDataset(records[n_eval:], model.config)
eval_ds = TTSDataset(records[:n_eval], model.config)
print("train:", len(train_ds), "eval:", len(eval_ds))
_ = train_ds[0]  # vérifie que le format est valide avant de lancer l'entraînement


def fast_codebook_logits(core, slow_hidden, codebooks):
    """Teacher forcing parallèle de la branche rapide (identique au SFT officiel)."""
    prefix = codebooks[:, :-1].long()
    fast_dtype = core.fast_embeddings.weight.dtype
    hidden = core.fast_project_in(slow_hidden.to(dtype=fast_dtype))
    hidden = torch.cat((hidden[:, None, :], core.fast_embeddings(prefix)), dim=1)
    length = int(hidden.shape[1])
    positions = torch.arange(length, device=hidden.device)
    attention_mask = torch.ones((hidden.shape[0], length), dtype=torch.long, device=hidden.device)
    mask = core._causal_mask(attention_mask, positions, length)
    rope = core.fast_freqs_cis[:length]
    for layer in core.fast_layers:
        if bool(getattr(core, "gradient_checkpointing", False)) and core.training:
            hidden = checkpoint(layer, hidden, rope, mask, use_reentrant=False)
        else:
            hidden = layer(hidden, rope, mask)
    return core.fast_output(core.fast_norm(hidden))


IM_END_ID = tokenizer.convert_tokens_to_ids("<|im_end|>")
assert isinstance(IM_END_ID, int) and IM_END_ID >= 0, "token <|im_end|> introuvable"


def slow_targets(core, tok0):
    """Tête lente compacte du 0.1B : logits [codebook_size + 1].
    indices 0..4095 = code sémantique (id - semantic_begin_id), index 4096 = EOS (<|im_end|>).
    Les autres positions (texte, '\\n' final) sont ignorées."""
    begin = int(core.config.semantic_begin_id)
    end = int(core.config.semantic_end_id)
    cb = int(core.config.codebook_size)
    target = torch.full_like(tok0, -100)
    is_sem = tok0.ge(begin) & tok0.le(end)
    target[is_sem] = tok0[is_sem] - begin
    target[tok0.eq(IM_END_ID)] = cb
    return target


def dual_loss(model, inputs):
    """Loss DualAR : slow (sémantique + EOS) + fast (codebooks). Réutilisée par le Trainer et le test CPU."""
    inputs = dict(inputs)
    labels = inputs.pop("labels")
    outputs = model(**inputs, return_dict=True)
    core = model.module if hasattr(model, "module") else model

    n_cls = outputs.logits.shape[-1]
    if n_cls == int(core.config.codebook_size) + 1:           # tête compacte (0.1B)
        slow_target = slow_targets(core, labels[:, 0])
    else:                                                     # tête sur tout le vocabulaire (0.6B)
        slow_target = labels[:, 0]
    slow_loss = F.cross_entropy(
        outputs.logits.reshape(-1, n_cls),
        slow_target.reshape(-1),
        ignore_index=-100,
    )
    sem = labels[:, 0].ge(int(core.config.semantic_begin_id)) & labels[:, 0].le(
        int(core.config.semantic_end_id)
    )
    slow_hidden = outputs.hidden_states[sem]
    if bool(core.config.norm_fastlayer_input):
        slow_hidden = core.norm(slow_hidden)
    codebooks = labels[:, 1:].permute(0, 2, 1)[sem]
    cb_logits = fast_codebook_logits(core, slow_hidden, codebooks)
    fast_loss = F.cross_entropy(cb_logits.reshape(-1, cb_logits.shape[-1]), codebooks.reshape(-1))
    return slow_loss + fast_loss, slow_loss, fast_loss, outputs


class Audio8Trainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        loss, slow_loss, fast_loss, outputs = dual_loss(model, inputs)
        if not torch.isfinite(loss):
            raise FloatingPointError(f"loss non finie : slow={slow_loss.item()} fast={fast_loss.item()}")
        if self.model.training and self.state.global_step % self.args.logging_steps == 0:
            self.log({"slow_loss": float(slow_loss), "fast_loss": float(fast_loss)})
        return (loss, {"logits": outputs.logits}) if return_outputs else loss

Durée totale encodée : 20.34 h
Paramètres entraînables : 169.8M / 169.8M
train: 4126 eval: 84


# **[6b] Test sur CPU AVANT l'entraînement**

In [18]:
# Sur GPU, un "device-side assert" masque la cause et corrompt la session ;
# sur CPU, Python donne le vrai message d'erreur. Le modèle est encore sur CPU ici.
cfg = model.config
print({k: getattr(cfg, k, None) for k in
       ["vocab_size", "semantic_begin_id", "semantic_end_id", "num_codebooks", "codebook_size",
        "pad_token_id", "max_seq_len"]})
print("codes  min / max :", min(int(r["codes"].min()) for r in records),
      "/", max(int(r["codes"].max()) for r in records))
dbg = collate([train_ds[i] for i in range(min(2, len(train_ds)))], int(cfg.pad_token_id))
print("input_ids max par ligne :", dbg["input_ids"].amax(dim=(0, 2)).tolist())
print("labels    max par ligne :", dbg["labels"].amax(dim=(0, 2)).tolist())
for n, prm in model.named_parameters():
    if any(k in n for k in ("embed", "output", "lm_head")) and "layers." not in n:
        print("  ", n, tuple(prm.shape))
model.eval()
with torch.no_grad():
    l, sl, fl, o = dual_loss(model, dbg)
st = slow_targets(model, dbg["labels"][:, 0])
print("cibles slow : min", int(st[st >= 0].min()), "| max", int(st.max()),
      "| nb EOS", int(st.eq(int(cfg.codebook_size)).sum()), "pour", dbg["labels"].shape[0], "exemples",
      "| <|im_end|> id =", IM_END_ID, "| config.eos_token_id =", getattr(cfg, "eos_token_id", None))
print(f"CPU OK | loss={l.item():.3f} slow={sl.item():.3f} fast={fl.item():.3f} | logits {tuple(o.logits.shape)}")

args = TrainingArguments(
    output_dir=OUT,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    num_train_epochs=EPOCHS,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    max_grad_norm=1.0,
    bf16=use_bf16,
    gradient_checkpointing=True,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=500,
    save_steps=500,
    save_total_limit=2,
    remove_unused_columns=False,
    dataloader_num_workers=2,
    report_to="none",
    push_to_hub=PUSH_CHECKPOINTS,
    hub_model_id=f"{HF_REPO}-ckpt" if PUSH_CHECKPOINTS else None,
    hub_strategy="checkpoint" if PUSH_CHECKPOINTS else "every_save",
    hub_private_repo=True,
)

pad_id = int(model.config.pad_token_id)
trainer = Audio8Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=lambda ex: collate(ex, pad_id),
)

{'vocab_size': 69633, 'semantic_begin_id': 65537, 'semantic_end_id': 69632, 'num_codebooks': 10, 'codebook_size': 4096, 'pad_token_id': 0, 'max_seq_len': 2048}
codes  min / max : 0 / 4095
input_ids max par ligne : [69621, 4084, 1014, 1023, 1023, 1023, 1023, 1022, 1022, 1021, 1023]
labels    max par ligne : [69621, 4084, 1014, 1023, 1023, 1023, 1023, 1022, 1022, 1021, 1023]
   slow.embed_tokens.weight (69633, 512)
   codebook_embeddings.weight (40960, 512)
   fast_embeddings.weight (4096, 512)
   fast_output.weight (4096, 512)
   semantic_output.weight (4097, 512)
cibles slow : min 2 | max 4096 | nb EOS 2 pour 2 exemples | <|im_end|> id = 228 | config.eos_token_id = 228
CPU OK | loss=14.867 slow=6.303 fast=8.564 | logits (2, 533, 4097)


# **[7] Entraînement  (test rapide : max_steps=20 dans TrainingArguments)**

In [ ]:
trainer.train(resume_from_checkpoint=RESUME_FROM)

/tmp/ipykernel_764/4020502782.py:94: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  self.log({"slow_loss": float(slow_loss), "fast_loss": float(fast_loss)})


Step,Training Loss,Validation Loss


# **[8] Sauvegarde du modèle final**

In [ ]:
FINAL = f"{OUT}/final"
core = trainer.model
core.gradient_checkpointing_disable()
core.config.use_gradient_checkpointing = False
for key, value in list(vars(core.config).items()):      # retire les champs non sérialisables
    try:
        json.dumps(value)
    except TypeError:
        delattr(core.config, key)
trainer.save_model(FINAL)
processor.save_pretrained(FINAL)

# Sécurité : copie les fichiers du checkpoint d'origine absents du dossier final (ex. codec).
snap = snapshot_download(MODEL_ID)
for root, _, files in os.walk(snap):
    for name in files:
        src = os.path.join(root, name)
        dst = os.path.join(FINAL, os.path.relpath(src, snap))
        if not os.path.exists(dst):
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.copy(src, dst)
print("Modèle sauvegardé dans", FINAL)

if UPLOAD_FINAL:
    from huggingface_hub import HfApi
    api = HfApi()
    api.create_repo(HF_REPO, private=True, exist_ok=True)
    api.upload_folder(folder_path=FINAL, repo_id=HF_REPO)
    print("Modèle envoyé sur https://huggingface.co/" + HF_REPO)

# **[9] Test d'inférence en malgache (même API que ton script de test)**

In [ ]:
del trainer, model
torch.cuda.empty_cache()

dtype = torch.bfloat16 if use_bf16 else torch.float32
processor = AutoProcessor.from_pretrained(FINAL, trust_remote_code=True)
tts = AutoModel.from_pretrained(FINAL, trust_remote_code=True, dtype=dtype).eval().to(device)

ref_text = open(f"{WORK}/ref.txt", encoding="utf-8").read()
inputs = processor(
    text=["""Indray andro, raha niara-nivory teo ambanin’ny hazo aviavy lehibe ireo biby, dia nanontany i Hanitriniala hoe:
— Iza aminareo no mahay zavatra mihoatra noho izaho? Jereo ange ny elatro e! Tsy misy vorona hafa tsara tahaka ahy eto amin’ity ala ity. Mahay manidina avo aho, mahay mihira tsara, ary mahita zavatra lavitra noho ny hainareo rehetra.
Nitsiky fotsiny i Sokina."""],
    reference_audio=[f"{WORK}/ref.wav"],
    reference_text=[ref_text],
    return_tensors="pt",
)
inputs = {k: v.to(device) for k, v in inputs.items()}
with torch.inference_mode():
    output = tts.generate(
        **inputs, max_new_tokens=512, temperature=0.7, top_p=0.9, top_k=50,
        do_sample=True, return_dict_in_generate=True,
    )
    waveforms, lengths = tts.decode_audio(output.codes)
audio = waveforms[0, : int(lengths[0])].float().cpu().numpy()
sf.write("test_malagasy.wav", audio, tts.config.codec_sample_rate)
print("test_malagasy.wav généré")